# Notebook 3 — Azure Databricks Pipeline Demo

This notebook demonstrates how the flatness clustering pipeline runs on
**Azure Databricks** with PySpark, MLflow, and Delta Lake.

## Architecture

```
Cold-rolling mill
     |  Flatness measurements (1 point/cm² × up to 15M points/coil)
     v
Delta Lake: steeldefect_flatness.raw_profiles
     |  01_feature_extraction.py  (PySpark pandas_udf)
     v
Delta Lake: steeldefect_flatness.coil_features   (105 features per coil — Groups A/B/C/D)
     |  02_clustering_training.py  (KMeans / GMM / Agglomerative / MiniBatchKMeans + MLflow)
     v
MLflow Model Registry: steeldefect_flatness_clusterer
Delta Lake: steeldefect_flatness.cluster_assignments
     |  03_online_assignment.py  (streaming / batch)
     v
Power BI / downstream RCA analysis
```

**Features (105-dim)**:
- Group A (0–29): Zonal statistics
- Group B (30–79): Cross-width snapshots
- Group C (80–94): Spectral (FFT)
- Group D (95–104): Temporal evolution (Saas et al. 2017 — autocorr, DTW, PAA)

**8 flatness pattern classes**: center_wave, edge_wave, symmetric_bow, quarter_buckle,
flat, random_noise, longitudinal_bow, complex_wave

> **Note:** Cells marked with `[DATABRICKS]` require the Databricks runtime with
> `spark`, `dbutils`, and `mlflow` pre-available.  All other cells run locally.

## Part 1 — Delta Lake Table Schema

The raw profiles are stored as binary blobs in Delta Lake.

In [1]:
# [DATABRICKS] Run this cell on a Databricks cluster
# Creates the raw_profiles table schema and shows a sample

raw_profiles_ddl = """
CREATE TABLE IF NOT EXISTS steeldefect_flatness.raw_profiles (
    coil_id        STRING         COMMENT 'Unique coil identifier, e.g. C00042',
    grade          STRING         COMMENT 'Steel grade, e.g. DC04, DP600',
    customer       STRING         COMMENT 'Customer code, e.g. GM_NA',
    timestamp      TIMESTAMP      COMMENT 'Mill measurement timestamp (UTC)',
    profile_blob   BINARY         COMMENT 'Raw little-endian float32 array (W*L values)',
    profile_width  INT            COMMENT 'Number of width measurement points',
    profile_length INT            COMMENT 'Number of length measurement points'
)
USING DELTA
PARTITIONED BY (DATE(timestamp))
COMMENT 'Raw flatness profiles from cold-rolling mill flatness gauge'
"""

# spark.sql(raw_profiles_ddl)  # Uncomment on Databricks
print("Table DDL (execute on Databricks):")
print(raw_profiles_ddl)

Table DDL (execute on Databricks):

CREATE TABLE IF NOT EXISTS steeldefect_flatness.raw_profiles (
    coil_id        STRING         COMMENT 'Unique coil identifier, e.g. C00042',
    grade          STRING         COMMENT 'Steel grade, e.g. DC04, DP600',
    customer       STRING         COMMENT 'Customer code, e.g. GM_NA',
    timestamp      TIMESTAMP      COMMENT 'Mill measurement timestamp (UTC)',
    profile_blob   BINARY         COMMENT 'Raw little-endian float32 array (W*L values)',
    profile_width  INT            COMMENT 'Number of width measurement points',
    profile_length INT            COMMENT 'Number of length measurement points'
)
USING DELTA
PARTITIONED BY (DATE(timestamp))
COMMENT 'Raw flatness profiles from cold-rolling mill flatness gauge'



## Part 2 — Write a Test Coil to Delta Lake

This simulates what the mill system would write when a coil is produced.

In [2]:
# [LOCAL] Works without Spark — shows the data structure

import sys
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("..").resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from flatness_clustering.data.generate_flatness_data import GENERATORS

rng = np.random.default_rng(7)
profile = GENERATORS["center_wave"](rng, width=150, length=2000)  # (150, 2000)

# Serialise to bytes (little-endian float32)
profile_blob = profile.astype("<f4").tobytes()

row = pd.DataFrame([{
    "coil_id":        "C99001",
    "grade":          "DP800",
    "customer":       "GM_NA",
    "timestamp":      "2024-06-15T08:32:00",
    "profile_blob":   profile_blob,
    "profile_width":  150,
    "profile_length": 2000,
}])

print(f"Coil row shape: {row.shape}")
print(f"profile_blob size: {len(profile_blob):,} bytes  ({len(profile_blob)/1e6:.2f} MB)")
print(f"Width × Length: {profile.shape}  →  {profile.size:,} float32 values")
print(f"\nRow (without blob):")
print(row.drop(columns=["profile_blob"]).to_string(index=False))

Coil row shape: (1, 7)
profile_blob size: 1,200,000 bytes  (1.20 MB)
Width × Length: (150, 2000)  →  300,000 float32 values

Row (without blob):
coil_id grade customer           timestamp  profile_width  profile_length
 C99001 DP800    GM_NA 2024-06-15T08:32:00            150            2000


## Part 3 — Feature Extraction (local simulation)

Simulates what `01_feature_extraction.py` does on each coil.

In [3]:
from flatness_clustering.features.statistical_features import (
    extract_statistical_features, feature_names,
)

feat = extract_statistical_features(profile)
names = feature_names()

print(f"Feature vector: {feat.shape}  ({len(feat)} features — 105 after Group D addition)")
print(f"\nFeature groups:")
print(f"  Group A (zonal stats):             features  0-29  (30 features)")
print(f"  Group B (cross-width snapshots):   features 30-79  (50 features)")
print(f"  Group C (spectral):                features 80-94  (15 features)")
print(f"  Group D (temporal evolution):      features 95-104 (10 features, Saas 2017)")

print(f"\nSample features (Group A — zonal stats):")
feat_df = pd.DataFrame({"name": names[:30], "value": feat[:30]})
print(feat_df.round(4).to_string(index=False))

print(f"\nSample features (Group D — temporal evolution):")
feat_df_d = pd.DataFrame({"name": names[95:], "value": feat[95:]})
print(feat_df_d.round(6).to_string(index=False))

Feature vector: (125,)  (125 features — 105 after Group D addition)

Feature groups:
  Group A (zonal stats):             features  0-29  (30 features)
  Group B (cross-width snapshots):   features 30-79  (50 features)
  Group C (spectral):                features 80-94  (15 features)
  Group D (temporal evolution):      features 95-104 (10 features, Saas 2017)

Sample features (Group A — zonal stats):
                          name     value
           zone_left_edge_mean -0.004500
            zone_left_edge_std  1.492800
    zone_left_edge_peak2valley 12.363000
            zone_left_edge_rms  1.492800
       zone_left_edge_skewness -0.007900
       zone_left_edge_kurtosis  0.009900
        zone_left_quarter_mean -0.023500
         zone_left_quarter_std  8.985100
 zone_left_quarter_peak2valley 65.244598
         zone_left_quarter_rms  8.985200
    zone_left_quarter_skewness  0.001200
    zone_left_quarter_kurtosis  1.832600
              zone_center_mean -0.097500
               zone_

## Part 4 — MLflow Experiment Structure

The clustering training job logs to MLflow.  Below is the expected run structure.

In [4]:
mlflow_structure = """
MLflow Experiment: /steeldefect/flatness_clustering

Run: clustering_kmeans_k6
  Parameters:
    n_clusters    = 6
    method        = kmeans    # also: gmm, agglomerative, minibatch_kmeans
    random_state  = 42
    n_coils       = 400       # 50 coils x 8 classes

  Metrics:
    silhouette_score       = 0.4231   (higher is better)
    davies_bouldin_index   = 0.8912   (lower is better)
    calinski_harabasz      = 4821.3   (higher is better)
    kmeans_inertia         = 12847.3

  Artifacts:
    models/clusterer.joblib          <- CoilClusterer (scaler + KMeans)
    sklearn_model/                   <- MLflow sklearn flavour
      MLmodel
      model.pkl
      conda.yaml
    plots/algorithm_comparison.png   <- KMeans/GMM/Agglomerative/MiniBatch vs k=3..8

  Tags:
    run_timestamp = 2024-06-15T09:00:00

Model Registry:
  Name:    steeldefect_flatness_clusterer
  Stage:   Production (latest run promoted)
  Version: 3
"""

print(mlflow_structure)


MLflow Experiment: /steeldefect/flatness_clustering

Run: clustering_kmeans_k6
  Parameters:
    n_clusters    = 6
    method        = kmeans    # also: gmm, agglomerative, minibatch_kmeans
    random_state  = 42
    n_coils       = 400       # 50 coils x 8 classes

  Metrics:
    silhouette_score       = 0.4231   (higher is better)
    davies_bouldin_index   = 0.8912   (lower is better)
    calinski_harabasz      = 4821.3   (higher is better)
    kmeans_inertia         = 12847.3

  Artifacts:
    models/clusterer.joblib          <- CoilClusterer (scaler + KMeans)
    sklearn_model/                   <- MLflow sklearn flavour
      MLmodel
      model.pkl
      conda.yaml
    plots/algorithm_comparison.png   <- KMeans/GMM/Agglomerative/MiniBatch vs k=3..8

  Tags:
    run_timestamp = 2024-06-15T09:00:00

Model Registry:
  Name:    steeldefect_flatness_clusterer
  Stage:   Production (latest run promoted)
  Version: 3



## Part 5 — Online Assignment (local simulation)

Simulates `03_online_assignment.py` assigning 6 new coils.

In [5]:
from flatness_clustering.pipeline.inference_pipeline import assign_new_coil

CLUSTERER_PATH = str(ROOT / "flatness_clustering" / "outputs" / "clusterer.joblib")
rng = np.random.default_rng(2024)

print("Simulating online assignment of new coils from the mill:")
print(f"{'Coil ID':10s}  {'Grade':8s}  {'Pattern':16s}  {'Cluster':8s}  {'Confidence':12s}  {'Outlier':8s}")
print("-" * 75)

# 8 test cases — includes the 2 new pattern classes
test_cases = [
    ("C10001", "DC04",  "center_wave"),
    ("C10002", "DP600", "edge_wave"),
    ("C10003", "DP800", "symmetric_bow"),
    ("C10004", "HC220", "quarter_buckle"),
    ("C10005", "DX54D", "flat"),
    ("C10006", "DC04",  "random_noise"),
    ("C10007", "DP800", "longitudinal_bow"),   # new class: parabolic bow along rolling direction
    ("C10008", "HC220", "complex_wave"),       # new class: mix of center + edge wave
]

if Path(CLUSTERER_PATH).exists():
    for coil_id, grade, pattern in test_cases:
        profile = GENERATORS[pattern](rng, width=150, length=2000)
        result = assign_new_coil(profile, clusterer_path=CLUSTERER_PATH)
        # DEC soft assignments: show top-2 clusters
        top2_idx = np.argsort(result["soft_assignments"])[-2:][::-1]
        top2_str = " | ".join(
            f"C{i}:{result['soft_assignments'][i]:.2f}" for i in top2_idx
        )
        print(
            f"{coil_id:10s}  {grade:8s}  {pattern:16s}  "
            f"{result['cluster_id']:8d}  "
            f"{result['confidence']:.4f}       "
            f"{'YES' if result['is_outlier'] else 'no':8s}  top2=[{top2_str}]"
        )
else:
    print(f"Clusterer not found at {CLUSTERER_PATH}")
    print("Run: python flatness_clustering/pipeline/train_pipeline.py")

Simulating online assignment of new coils from the mill:
Coil ID     Grade     Pattern           Cluster   Confidence    Outlier 
---------------------------------------------------------------------------
C10001      DC04      center_wave              3  0.3703       no        top2=[C3:0.37 | C1:0.17]
C10002      DP600     edge_wave                3  0.3091       no        top2=[C3:0.31 | C0:0.19]
C10003      DP800     symmetric_bow            4  0.4099       no        top2=[C4:0.41 | C0:0.14]
C10004      HC220     quarter_buckle           3  0.3216       no        top2=[C3:0.32 | C0:0.17]
C10005      DX54D     flat                     1  0.5829       no        top2=[C1:0.58 | C3:0.10]
C10006      DC04      random_noise             0  0.2602       no        top2=[C0:0.26 | C3:0.19]
C10007      DP800     longitudinal_bow         4  0.4480       no        top2=[C4:0.45 | C0:0.13]
C10008      HC220     complex_wave             3  0.3822       no        top2=[C3:0.38 | C1:0.17]


## Part 6 — Databricks Job Configuration

Recommended job structure for production deployment.

In [6]:
job_config = """
Databricks Jobs Configuration (JSON excerpt)
=============================================

Job 1: Daily Feature Extraction
  schedule: 0 2 * * *  (daily at 02:00 UTC)
  cluster:  Standard_DS3_v2 x 4 workers
  notebook: /steeldefect/flatness_clustering/databricks/01_feature_extraction
  parameters:
    input_table:  steeldefect_flatness.raw_profiles
    output_table: steeldefect_flatness.coil_features

Job 2: Weekly Clustering Retraining
  schedule: 0 4 * * 0  (weekly Sunday 04:00 UTC)
  cluster:  Standard_DS3_v2 x 2 workers
  notebook: /steeldefect/flatness_clustering/databricks/02_clustering_training
  parameters:
    n_clusters:    6
    method:        kmeans

Job 3: Continuous Online Assignment
  trigger:  Delta Live Tables (file arrival)
  cluster:  Standard_DS3_v2 x 2 workers  (auto-scaling)
  notebook: /steeldefect/flatness_clustering/databricks/03_online_assignment
  parameters:
    mode:    streaming

Storage:
  /dbfs/steeldefect/flatness_clustering/
    models/clusterer.joblib     <- trained model artifact
    checkpoints/assignment/     <- streaming checkpoints
"""

print(job_config)


Databricks Jobs Configuration (JSON excerpt)

Job 1: Daily Feature Extraction
  schedule: 0 2 * * *  (daily at 02:00 UTC)
  cluster:  Standard_DS3_v2 x 4 workers
  notebook: /steeldefect/flatness_clustering/databricks/01_feature_extraction
  parameters:
    input_table:  steeldefect_flatness.raw_profiles
    output_table: steeldefect_flatness.coil_features

Job 2: Weekly Clustering Retraining
  schedule: 0 4 * * 0  (weekly Sunday 04:00 UTC)
  cluster:  Standard_DS3_v2 x 2 workers
  notebook: /steeldefect/flatness_clustering/databricks/02_clustering_training
  parameters:
    n_clusters:    6
    method:        kmeans

Job 3: Continuous Online Assignment
  trigger:  Delta Live Tables (file arrival)
  cluster:  Standard_DS3_v2 x 2 workers  (auto-scaling)
  notebook: /steeldefect/flatness_clustering/databricks/03_online_assignment
  parameters:
    mode:    streaming

Storage:
  /dbfs/steeldefect/flatness_clustering/
    models/clusterer.joblib     <- trained model artifact
    checkpoin

## Summary

| Component | Technology | Purpose |
|-----------|-----------|--------|
| **Raw profiles** | Delta Lake (BINARY columns) | Store compressed float32 arrays |
| **Feature extraction** | PySpark `pandas_udf` | Parallel extraction across cluster |
| **Feature storage** | Delta Lake (95 DOUBLE columns) | Query-ready feature table |
| **Model training** | MLflow run | Reproducible, versioned clustering |
| **Model registry** | MLflow Model Registry | Production/Staging model management |
| **Online assignment** | Structured Streaming | Near-real-time coil classification |
| **Output** | Delta Lake cluster_assignments | Input for RCA / Power BI dashboards |

### Key advantages over batch processing

- **Streaming mode**: new coil assigned within seconds of leaving the mill
- **Delta Lake ACID**: no duplicate assignments even on cluster restart
- **MLflow versioning**: can rollback to previous clustering model if quality drops
- **Outlier detection**: coils with high cluster distance are flagged for review

## Sensitivity Sampling (Axiotis et al. 2024 Theorem 1)

Sensitivity-based coreset selection provably preserves the average loss up to $(1\pm\varepsilon)$:

$$s(x_i) = \frac{d(x_i, \mu_{C(i)})^2}{\Phi_k} + \frac{1}{|C(i)|}$$

where:
- $\mu_{C(i)}$ = centroid of $x_i$'s cluster
- $\Phi_k = \sum_i d(x_i, \mu_{C(i)})^2$ = total k-means cost
- $|C(i)|$ = cluster size

Points that are **outliers within their cluster** (large $d^2/\Phi_k$) AND **in rare clusters** ($1/|C_i|$ large) get the highest sensitivity scores.

In [7]:
import numpy as np
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

# Sensitivity sampling from Axiotis et al. (2024) — available in steeldefect_active_learning
from steeldefect_active_learning.active_learning.clustering import sensitivity_sample

np.random.seed(42)
# Simulate 200 coil feature embeddings with 5 unequal clusters
CLUSTER_SIZES = [40, 60, 20, 50, 30]
embeddings = {}
for cluster, n in enumerate(CLUSTER_SIZES):
    center = np.random.randn(105) * 5   # 105-dim feature space
    for i in range(n):
        key = f"cluster{cluster}_coil{i:03d}"
        embeddings[key] = (center + np.random.randn(105)).astype(np.float32)

# Select top 20 representative coils using sensitivity sampling
# s(x_i) = d(x_i, mu)^2 / Phi_k + 1/|C_i|
# Outliers in small clusters get highest scores
selected = sensitivity_sample(embeddings, n_clusters=5, budget=20, random_state=42)

# Analyze which clusters got selected
from collections import Counter
cluster_counts = Counter(k.split("_coil")[0] for k in selected)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: selected per cluster vs cluster size
clusters = [f"cluster{i}" for i in range(5)]
counts   = [cluster_counts.get(c, 0) for c in clusters]
labels   = [f"cluster{i}\n(n={CLUSTER_SIZES[i]})" for i in range(5)]
colors   = ["#2196F3","#4CAF50","#FF9800","#E91E63","#9C27B0"]
axes[0].bar(labels, counts, color=colors)
axes[0].set_title("Sensitivity Sampling: Selected coils per cluster\n"
                  "Axiotis et al. (2024) — s(x) = d^2/Phi_k + 1/|C_i|")
axes[0].set_ylabel("# coils selected (budget=20)")
for i, c in enumerate(counts):
    axes[0].text(i, c + 0.1, f"{c/20:.0%}", ha="center", fontsize=10)

# Right: budget fraction vs cluster size fraction
budget_frac = [c/20 for c in counts]
size_frac   = [s/sum(CLUSTER_SIZES) for s in CLUSTER_SIZES]
axes[1].scatter(size_frac, budget_frac, s=200, c=colors, zorder=5)
axes[1].plot([0,0.35],[0,0.35], "k--", alpha=0.4, label="Proportional baseline")
for i, (sf, bf) in enumerate(zip(size_frac, budget_frac)):
    axes[1].annotate(f"C{i}", (sf, bf), textcoords="offset points",
                     xytext=(5,5), fontsize=9)
axes[1].set_xlabel("Cluster size fraction")
axes[1].set_ylabel("Budget fraction allocated")
axes[1].set_title("Sensitivity vs Uniform Allocation\n"
                  "Small clusters get proportionally more budget")
axes[1].legend()

plt.tight_layout()
plt.savefig("sensitivity_sampling.png", dpi=120)
print(f"Top 10 representative coils: {selected[:10]}")
print(f"\nCluster allocation: {dict(zip(range(5), counts))}")
print("Saved: sensitivity_sampling.png")

Top 10 representative coils: ['cluster3_coil041', 'cluster2_coil002', 'cluster4_coil007', 'cluster3_coil023', 'cluster0_coil018', 'cluster4_coil026', 'cluster3_coil038', 'cluster3_coil044', 'cluster0_coil025', 'cluster2_coil003']

Cluster allocation: {0: 3, 1: 2, 2: 3, 3: 7, 4: 5}
Saved: sensitivity_sampling.png
